# 05. 최종 요약

## [ 프로젝트 개요 ]

본 프로젝트는 한국미디어패널(KMP) 2020~2025 개인조사 데이터를 활용하여 통신사 고객 이탈 가능성을 예측하는 것을 목표로 하였다.
예측 대상은 두 가지 라벨로 정의하였다.

- `churn_any`: 전년 대비 이용 통신사가 변경되었는지 여부
- `churn_to_mvno`: 전년 메이저 통신사(1/2/3) 이용자가 다음 해 알뜰폰(MVNO=4)으로 이동했는지 여부

두 라벨은 모두 통신사 이동과 관련되어 있지만, 발생 빈도와 예측 난이도에 큰 차이가 있다.
따라서 본 프로젝트에서는 두 문제를 분리하여 baseline 성능, 개선 가능성, 해석 방향을 각각 확인하였다.

---

## [ 데이터 및 전처리 ]

분석에는 KMP 2020~2025 개인조사 CSV(`p20 ~ p25`)와 코드북(`P_codebook_v32.xlsx`)을 사용하였다.
데이터는 `pid` 기준으로 전년과 다음 해를 연결한 transition 기반 long panel 형태로 재구성하였다.
즉, 한 행은 `(pid, year_t0=t-1 → year_t1=t)` 구조를 가지며, 입력 변수는 항상 `t-1` 시점 정보만 사용하였다.

전처리 과정에서는 다음 기준을 적용하였다.

- 결측 코드 `9999`, `9998`, `9997`를 `NaN`으로 변환
- 공백 및 NBSP 제거
- 통신사 변수는 유효값 `{1, 2, 3, 4}`만 유지
- 일부 이진 변수 `1/2 → 1/0`으로 변환
- `pid`, `year_t0`, `year_t1` 기준 중복 제거
- 예측에는 `t-1` 시점 변수만 사용하고 `telco` 관련 정보는 누수 방지를 위해 제외

최종 데이터셋은 **41,299행, 17컬럼**으로 구성되었으며, 전처리 후 결측과 중복은 남지 않았다.

---

## [ 사용 변수 ]

분석에는 총 10개 변수군을 사용하였다.

- 스마트폰 구분
- 음성 무제한 서비스 가입 여부
- 데이터 무제한 서비스 가입 여부
- 월평균 휴대폰 이용 총 금액
- 월평균 기기 할부금
- 휴대폰 결합상품 가입 여부
- 휴대폰 요금 부담자
- 나이 또는 연령대
- 개인 월평균 소득
- 직업 유무

기본 hold-out baseline 분석인 `02`, `03`에서는 `나이(age1_tminus1)`를 직접 사용하였다.
반면 `04`의 churn_any 교차검증·튜닝 단계에서는 이를 `연령대(age band)`로 재구성하여 해석하였다.

즉 기존 통신 이용 특성 변수에 더해 개인의 생애 주기와 경제적 배경까지 함께 반영할 수 있도록 분석 구조를 확장하였다.

---

## [ EDA 및 문제 구조 ]

EDA와 전처리 점검 결과, 라벨 분포는 다음과 같이 확인되었다.

- `churn_any`: **14,982 / 41,299 ≈ 36.3%**
- `churn_to_mvno`: **515 / 41,299 ≈ 1.25%**

즉 `churn_any`는 상대적으로 표본이 충분한 문제였고,
`churn_to_mvno`는 매우 희소한 이벤트로서 예측 난이도가 훨씬 높았다.

또한 변수 분포와 이탈률 비교를 통해 비용 관련 변수, 서비스 가입 여부, 단말 특성, 나이/연령대와 소득 등에서 라벨별 분포 차이가 관찰되었으며, 이는 이후 baseline 모델에서 함께 확인할 필요가 있는 후보 변수임을 시사하였다.

---

## [ churn_any 분석 결과 ]

`churn_any`는 일반적인 통신사 변경 여부를 예측하는 문제로, hold-out baseline 비교와 GroupKFold 기반 교차검증·튜닝을 함께 진행하였다.

먼저 `02`의 단일 hold-out baseline 비교에서는 `DecisionTree`가 Recall `0.8646`, F1 `0.5265`로 가장 강한 탐지형 baseline 모습을 보였다.
반면 `RandomForest`는 Accuracy `0.5739`, Precision `0.4112`, Recall `0.4794`, F1 `0.4427`, PR-AUC `0.4090`을 기록하여
`DecisionTree`보다 더 보수적이지만 Accuracy와 Precision 측면에서는 상대적으로 안정적인 대안으로 해석할 수 있었다.
또한 `LogisticRegression`은 Recall `0.5588`, F1 `0.4534`로 중간형 기준선 역할을 수행하였다.
반면 `GradientBoosting`과 `XGBoost`는 Accuracy는 높게 보일 수 있었지만 Recall이 매우 낮아 churn 고객 탐지 관점에서는 실용성이 떨어졌다.

다만 이 hold-out baseline 결과는 단일 분할 기준이므로, `04`의 GroupKFold 평균 결과와 직접 같은 선상에서 비교하면 안 된다.

이후 `04`의 GroupKFold 기반 교차검증에서는 baseline 평균 성능 기준으로 `DecisionTree`가 Recall `0.8244`, F1 `0.5266`으로 가장 높은 성능을 보였다.
`RandomForest`는 baseline 절대 1위는 아니었지만 ROC-AUC와 PR-AUC 측면에서 비교적 안정적이었고, 튜닝을 통해 개선 가능성이 있다고 판단하여 주요 조정 대상으로 선정하였다.

최적 RandomForest 조합은 다음과 같았다.

- `n_estimators=300`
- `max_depth=8`
- `min_samples_split=10`
- `min_samples_leaf=3`

튜닝 결과, `RandomForest`의 성능은 baseline 대비 분명히 개선되었다.

- F1: 0.4608 → 0.5187
- Recall: 0.5230 → 0.7262

또한 hold-out 기준 threshold를 비교한 결과,
임계값 `0.45`에서 Precision `0.3819`, Recall `0.8868`, F1 `0.5339`로 가장 좋은 균형을 보였다.

변수 중요도는 원변수 기준으로 다시 합산하여 해석하였다.
그 결과 `스마트폰 구분`, `월평균 휴대폰 이용 총 금액`, `월평균 기기 할부금`, `개인 월평균 소득`, `연령대`가 상대적으로 중요한 변수로 나타났다.
특히 `스마트폰 구분`은 세부 범주별 중요도가 합쳐진 결과이므로, 특정 값 하나보다 스마트폰 구분 관련 정보 전체가 중요한 신호로 작용했다고 보는 것이 적절하다.

종합하면 `churn_any`는 단말 특성, 통신 이용 특성, 비용 부담, 경제적 여건, 나이/연령대가 함께 작용하는 문제로 해석할 수 있으며,
최종적으로는 `02`의 hold-out baseline에서는 `DecisionTree`가 가장 강한 탐지형 baseline, `04`의 GroupKFold baseline 최고 성능도 `DecisionTree`,
그리고 튜닝 후 실제 활용 가능성이 높은 모델은 `RandomForest`로 정리하는 것이 가장 정확하다.

---

## [ churn_to_mvno 분석 결과 ]

`churn_to_mvno`는 전체 데이터 기준 양성 비율이 약 `1.25%`에 불과한 매우 희소한 문제였다.
다만 `03`의 hold-out test split 기준 양성 비율은 약 `1.01%`로, 실제 baseline 해석에서는 이 수치를 함께 구분해서 보는 것이 적절하다.
따라서 Accuracy보다 Recall, F1, PR-AUC를 중심으로 해석하는 것이 적절하다.

baseline 비교에서는 `LogisticRegression`과 `DecisionTree`가 실제 양성을 상대적으로 더 많이 탐지하였다.

- `LogisticRegression`: Recall `0.5476`, F1 `0.0277`, ROC-AUC `0.6145`, PR-AUC `0.0189`
- `DecisionTree`: Recall `0.5357`, F1 `0.0276`, ROC-AUC `0.5923`, PR-AUC `0.0134`

즉 현재 baseline 기준에서는 `LogisticRegression`이 가장 안정적인 기준선 역할을 했고,
`DecisionTree`는 유사한 탐지형 대안으로 해석할 수 있었다.

반면 `RandomForest`는 Accuracy `0.9753`으로 매우 높게 보였지만 Recall `0.0238`, F1 `0.0191`에 그쳐
실제 알뜰폰 이동자를 거의 잡지 못했다.
`GradientBoosting`과 `XGBoost`도 기본 설정에서는 양성 탐지를 거의 수행하지 못해
현재처럼 1% 수준의 극심한 불균형 문제에서는 복잡한 부스팅 계열이 baseline에서 바로 강점을 보이지 못했다.

Threshold 조정도 함께 확인했지만, 단순 threshold 튜닝만으로는 희소 클래스의 근본적인 한계를 넘기 어려웠다.
예를 들어 `LogisticRegression`에서 F1 기준 최적 threshold를 적용해도 최고 F1은 `0.0402` 수준에 머물렀다.
이는 이탈 의심 고객을 더 넓게 잡을 수는 있어도 오탐지가 매우 많아 실제 운영에는 부담이 크다는 뜻이다.

또한 `EasyEnsemble` 같은 불균형 특화 기법도 추가 실험으로 확인했지만,
Recall `0.5952`, F1 `0.0313` 수준으로 일부 보완 가능성은 보였으나 정밀도와 전체 실용성 측면에서는 여전히 한계가 컸다.
따라서 이 결과는 baseline 자체를 대체하는 최종 결론이 아니라 불균형 대응 방향을 확인하는 참고용 결과로 해석하는 것이 적절했다.

종합하면 `churn_to_mvno`는 변수 추가만으로 성능이 크게 뛰는 문제가 아니라,
극심한 클래스 불균형 자체가 가장 큰 난점이며,
실무적으로는 threshold 조정, 불균형 특화 기법, 추가 행동·계약 변수 도입이 더 중요한 문제라고 정리할 수 있다.

---

## [ 공통 인사이트 ]

두 문제를 함께 보면 공통적으로 비용 및 이용 특성이 중요한 축으로 작용했다.
특히 `월평균 휴대폰 이용 총 금액`, `월평균 기기 할부금`, `스마트폰 구분`은 반복적으로 중요한 변수로 나타났다.

여기에 `나이/연령대`, `개인 월평균 소득`, `직업 유무`를 추가하면서,
이번 프로젝트는 단순한 통신 사용 패턴 분석을 넘어
“누가 더 이동 가능성이 높은가”를 설명할 수 있는 구조로 확장되었다.

즉 고객 이탈은 단순한 서비스 이용 행태만으로 설명되지 않으며,
비용 관련 변수와 단말·이용 특성이 공통 핵심 축으로 작용했고,
여기에 나이/연령대, 소득, 직업 유무 같은 개인 배경 변수가 해석 가능성을 확장해 주었다고 볼 수 있다.

---

## [ 한계 및 향후 개선 ]

첫째, 현재 변수 구조는 핵심 변수 중심의 제한된 구성이다.
실제 통신사 데이터에 포함될 수 있는 약정 만료 정보, 단말기 교체 주기, 최근 사용량 변화, 고객센터 접촉 이력, 프로모션 반응 이력 등은 반영하지 못했다.

둘째, `churn_to_mvno`는 양성 비율이 약 1.25%에 불과한 매우 희소한 문제이므로, 현재 구조만으로는 예측 성능에 명확한 한계가 있다.

셋째, 현재 분석은 `t-1` 시점의 정적 변수 중심이기 때문에, 시간에 따른 변화량이나 행동 추세를 충분히 반영하지 못한다.

---

## [ 최종 결론 ]

이번 프로젝트에서는 통신사 고객 이탈을 `churn_any`와 `churn_to_mvno`로 나누어 분석하였다.

`churn_any`는 상대적으로 표본이 충분한 문제였으며, baseline 비교와 RandomForest 튜닝을 통해 실제 활용 가능한 수준의 예측 성능과 해석 가능한 주요 변수군을 확보할 수 있었다.
특히 비용/이용 특성에 더해 나이/연령대와 소득 같은 개인 배경 변수를 반영함으로써, 이탈 현상을 더 현실적으로 해석할 수 있게 되었다.

반면 `churn_to_mvno`는 매우 희소한 문제로, 단순한 baseline과 threshold 조정만으로는 실질적인 성능 개선에 한계가 있었다.
따라서 이 문제는 성능 자체보다도 데이터 구조와 희소 클래스의 한계를 이해하고, 향후 불균형 특화 기법과 추가 변수 도입 필요성을 확인했다는 점에서 의미가 있다.

즉 이번 프로젝트의 핵심 성과는 단순히 모델 점수를 비교하는 데 그치지 않고,
통신사 이탈을 비용 관련 변수, 이용 특성, 개인 배경이 함께 작동하는 문제로 해석할 수 있는 분석 구조를 정리했다는 데 있다.
